In [1]:
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker
from db_manager_v2 import engine, Base, Security, get_session, Security, Investment, execute_sell_percentage_investment
from db_manager_v2 import get_available_gas_lamports, get_oldest_gas_security
from global_values import WORLD_STABLE_COIN, solana_tokens

2026-07-01 23:04:07 | INFO     | Logging initialized. Writing to logs/trades.log


In [10]:
# see gas
with get_session(engine) as session:
    gas = get_available_gas_lamports(session, wallet_pk=1)
    oldest = get_oldest_gas_security(session, wallet_pk=1)
    print("Available gas:", gas)
    print("Oldest gas ID:", oldest)
    

Available gas: Ok(20543265)
Oldest gas ID: Ok(16)


In [3]:
# This code shows open investments
with get_session(engine) as session:
    investments = session.execute(
        select(Investment).where(Investment.isClosed == False)
    ).scalars().all()
    
    print(f"Open Investments found: {len(investments)}")
    for inv in investments:
        print(f"ID: {inv.id} | Parent Asset: {inv.parent_id} | Amount: {inv.amount} | Buy TX: {inv.buy_tx_id}")

Open Investments found: 2
ID: 2 | Parent Asset: 9 | Amount: 104 | Buy TX: allocate_from_security_5
ID: 15 | Parent Asset: 7 | Amount: 9087533 | Buy TX: merge_usdc_investments


In [6]:
# This code trades an investment to the target mint
with get_session(engine) as session:
    result = execute_sell_percentage_investment(
        session=session,
        investment_id=15,
        target_mint="AZsHEMXd36Bj1EMNXhowJajpUXzrKcK57wW4ZGXVa7yR",
        sell_percentage=50.0,
        slippage_bps=310,
        gas_security_id=15,
        estimated_gas_lamports=10_000 #200_000
    )
    
    print(result)
    session.expire_all()

# 2. Immediately verify inside the same session/transaction
    investments = session.execute(
        select(Investment)
    ).scalars().all()

    print(f"\nTotal investments in DB: {len(investments)}")
    for inv in investments:
        print(f"ID: {inv.id} | Amount: {inv.amount} | Closed: {inv.isClosed}")

2026-07-01 23:07:12 | INFO     | [TRADE] Starting trade | investment_id=15
2026-07-01 23:07:13 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 23:07:13 | INFO     | [RPC] Using PRIMARY: https://mainnet.helius-rpc.com/
2026-07-01 23:07:13 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 23:07:14 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 23:07:14 | INFO     | STATUS: 200
2026-07-01 23:07:14 | INFO     | BODY: {"inputMint":"EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v","inAmount":"4543767","outputMint":"AZsHEMXd36Bj1EMNXhowJajpUXzrKcK57wW4ZGXVa7yR","outAmount":"88403888084985","otherAmountThreshold":"85663367554351","swapMode":"ExactIn","slippageBps":310,"platformFee":null,"priceImpactPct":"0","routePlan":[{"swapInfo":

Ok({'status': 'success', 'investment_id': 15, 'tx_signature': '5FZJv3aYXVk7AZtZfnxa9WoMHBSgTFEDxpNMCwCbhwUKozvbSVkAk2fLDUgdVtHueBR4S8ysGAAEAx3uJdYheqEr', 'sold_lamports': 4543767, 'new_investment_id': 16, 'sell_price_usdc': 0.0, 'priority_fee_lamports': 25000})

Total investments in DB: 17
ID: 2 | Amount: 104 | Closed: False
ID: 1 | Amount: 3409234 | Closed: True
ID: 4 | Amount: 14456598 | Closed: True
ID: 5 | Amount: 3409233 | Closed: True
ID: 6 | Amount: 3429188 | Closed: True
ID: 3 | Amount: 22863250 | Closed: True
ID: 7 | Amount: 6838421 | Closed: True
ID: 8 | Amount: 2290730 | Closed: True
ID: 9 | Amount: 4564576 | Closed: True
ID: 11 | Amount: 4564575 | Closed: True
ID: 12 | Amount: 228717539 | Closed: True
ID: 10 | Amount: 131488280773 | Closed: True
ID: 13 | Amount: 4554775 | Closed: True
ID: 14 | Amount: 4532758 | Closed: True
ID: 16 | Amount: 88403806367280 | Closed: False
ID: 15 | Amount: 4543767 | Closed: True
ID: 17 | Amount: 4543766 | Closed: False


In [7]:
with get_session(engine) as session:
    # read table data using sql query
    sql_df = pd.read_sql(
        "SELECT * FROM investment_table",
        con=engine
    )
    
    print(sql_df)

    id  parent_id          amount  purchase_price_usdc  sale_price_usdc  \
0    2          9             104             0.000000              NaN   
1    1          7         3409234             0.000000         0.000000   
2    4         15        14456598             3.408385         3.429188   
3    5          7         3409233             0.000000              NaN   
4    6          7         3429188             3.427953              NaN   
5    3          6        22863250             0.000000         2.290730   
6    7          7         6838421             0.000000              NaN   
7    8          7         2290730             2.289882              NaN   
8    9          7         4564576             0.000000         0.000000   
9   11          7         4564575             0.000000         0.000000   
10  12         17       228717539             4.562039         4.554775   
11  10         16    131488280773             4.553795         4.532758   
12  13          7        

In [8]:
solana_tokens

['DtR4D9FtVoTX2569gaL837ZgrB6wNjj6tkmnX9Rdk9B2',
 '8bd3PSBp15xbjjCJzEJDrQDPiBTXgAJtgoxTcmXxytWL',
 '61V8vBaqAGMpgDQi4JcAwo1dmBGHsyhzodcPqnEVpump',
 '9PR7nCP9DpcUotnDPVLUBUZKu5WAYkwrCUx9wDnSpump',
 '3B5wuUrMEi5yATD7on46hKfej3pfmd7t1RKgrsN3pump',
 'Ez3nzG9ofodYCvEmw73XhQ87LWNYVRM2s7diB5tBZPyM',
 '4vMsoUT2BWatFweudnQM1xedRLfJgJ7hswhcpz4xgBTy',
 'UXPhBoR3qG4UCiGNJfV7MqhHyFqKN68g45GoYvAeL2M',
 'METAewgxyPbgwsseH8T16a39CQ5VyVxZi9zXiDPY18m',
 'CaGa7pddFXS65Gznqwp42kBhkJQdceoFVT7AQYo8Jr8Q',
 'CTJf74cTo3cw8acFP1YXF3QpsQUUBGBjh2k2e8xsZ6UL',
 '5LafQUrVco6o7KMz42eqVEJ9LW31StPyGjeeu5sKoMtA',
 'HZ1JovNiVvGrGNiiYvEozEVgZ58xaU3RKwX8eACQBCt3',
 'JUPyiwrYJFskUPiHa7hkeR8VUtAeFoSYbKedZNsDvCN',
 'MEW1gQWJ3nEXg2qgERiKu7FAFj79PHvQVREQUzScPP5',
 'EKpQGSJtjMFqKZ9KQanSqYXRcF8fBopzLHYxdM65zcjm',
 'rndrizKT3MK1iimdxRdWabcF7Zg7AR5T4nud4EkHBof',
 'jtojtomepa8beP8AuQc6eXt5FriJwfFMwQx2v2f9mCL',
 '85VBFQZC9TZkfaptBWjvUw7YbZjy52A6mjtPGjstQAmQ',
 'ZEUS1aR7aX8DFFJf5QjWj2ftDDdNTroMNGo8YoQm3Gq',
 '4k3Dyjzvzp8eMZWUXbBCjEvwS

In [9]:
with get_session(engine) as session:
    # read table data using sql query
    sql_df = pd.read_sql(
        "SELECT * FROM security_table",
        con=engine
    )
    
    print(sql_df)

    id  parent_id    amount  purchase_price_usdc  sale_price_usdc  \
0    3         12         0                  0.0              NaN   
1    4         11         0                  0.0              NaN   
2    6         13         0                  0.0              NaN   
3    7         14         0                  0.0              NaN   
4    8         10         0                  0.0              NaN   
5    9          4         0                  0.0              NaN   
6   10          3         0                  0.0              NaN   
7   11          5         0                  0.0              NaN   
8   12          8         0                  0.0              NaN   
9   14          2         0                  0.0              NaN   
10   1          1         0                  0.0              NaN   
11   2          7         0                  0.0              NaN   
12   5          9         0                  0.0              NaN   
13  13          6         0       